In [1]:
#optimized search algorithm with GUI - Result of 1st Sprint
#

import numpy as np
import cv2
import matplotlib.pyplot as plt
import time
from collections import deque
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk

def trace_boundary(binary_image):
    """Finds and traces all boundary pixels in a binary image using BFS with 8-connectivity."""
    start_time = time.time()
    
    h, w = binary_image.shape
    boundary_mask = np.zeros((h, w), dtype=np.uint8)  # 0 = not boundary, 255 = boundary
    visited = np.zeros((h, w), dtype=bool)  # Track visited pixels
    labels = np.zeros((h, w), dtype=int)  # Store labels of each boundary pixel
    current_label = 1  # Start labeling from 1

    row_sums = np.sum(binary_image, axis=1)
    col_sums = np.sum(binary_image, axis=0)
    
    top = np.argmax(row_sums > 0)
    bottom = h - np.argmax(row_sums[::-1] > 0)
    left = np.argmax(col_sums > 0)
    right = w - np.argmax(col_sums[::-1] > 0)
    
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]

    def bfs(start_i, start_j):
        queue = deque([(start_i, start_j)])
        
        while queue:
            i, j = queue.popleft()
            
            if visited[i, j]:
                continue
            
            visited[i, j] = True
            boundary_mask[i, j] = 255
            labels[i, j] = current_label

            for dx, dy in directions:
                ni, nj = i + dx, j + dy
                if 0 <= ni < h and 0 <= nj < w and not visited[ni, nj]:
                    if binary_image[ni, nj] == 255:
                        boundary_found = any(0 <= ni + dx2 < h and 0 <= nj + dy2 < w and binary_image[ni + dx2, nj + dy2] == 0 for dx2, dy2 in directions)
                        if boundary_found:
                            queue.append((ni, nj))
    
    for i in range(top, bottom):
        for j in range(left, right):
            if binary_image[i, j] == 255 and not visited[i, j]:
                if any(0 <= i + dx < h and 0 <= j + dy < w and binary_image[i + dx, j + dy] == 0 for dx, dy in directions):
                    bfs(i, j)
                    current_label += 1

    print("--- %s seconds ---" % (time.time() - start_time))
    return boundary_mask, labels, current_label - 1


def compute_circularity(labels, num_labels):
    circularities = {}

    for label in range(1, num_labels + 1):
        # Get coordinates of all pixels with this label
        coords = np.column_stack(np.where(labels == label))
        
        if coords.shape[0] < 2:
            continue  # Skip tiny components
        
        # Compute centroid
        centroid = np.mean(coords, axis=0)

        # Compute Euclidean distances to centroid
        distances = np.linalg.norm(coords - centroid, axis=1)

        # Compute mean and variance
        mean_dist = np.mean(distances)
        var_dist = np.var(distances)

        # Avoid division by zero
        circularity = mean_dist / var_dist if var_dist != 0 else 0

        circularities[label] = circularity

    return circularities

def process_image():
    image_path = entry_path.get()
    erosion_iters = int(entry_iterations.get())
    border_thickness = int(entry_thickness.get())
    
    beans_img = cv2.imread(image_path)
    img_gray = cv2.cvtColor(beans_img, cv2.COLOR_BGR2GRAY)
    
    height, width = img_gray.shape
    border_x = int(height * 0.02)
    border_y = int(width * 0.02)
    img_cropped = img_gray[border_x:height-border_x, border_y:width-border_y]
    
    threshold = 75
    _, img_segmented = cv2.threshold(img_cropped, threshold, 255, cv2.THRESH_BINARY_INV)
    
    kernel_erode = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    img_eroded = cv2.erode(img_segmented, kernel_erode, iterations=erosion_iters)
    
    kernel_open = np.ones((3,3), np.uint8)
    img_opened = cv2.morphologyEx(img_eroded, cv2.MORPH_OPEN, kernel_open)
    
    boundary_mask, labels, num_labels = trace_boundary(img_opened)
    colored_img = cv2.cvtColor(img_opened, cv2.COLOR_GRAY2BGR)
    boundary_mask, labels, num_labels = trace_boundary(img_opened)
    circularities = compute_circularity(labels, num_labels)
    print("Circularities per label:", circularities)
    
    for i in range(border_thickness):
        dilated_mask = cv2.dilate(boundary_mask, np.ones((3,3), np.uint8), iterations=i)
        colored_img[dilated_mask == 255] = [0, 0, 255]
    
    img_original = Image.open(image_path)
    img_original.thumbnail((250, 250))
    img_tk = ImageTk.PhotoImage(img_original)
    label_original.config(image=img_tk)
    label_original.image = img_tk
    
    img_result = Image.fromarray(cv2.cvtColor(colored_img, cv2.COLOR_BGR2RGB))
    img_result.thumbnail((250, 250))
    img_result_tk = ImageTk.PhotoImage(img_result)
    label_result.config(image=img_result_tk)
    label_result.image = img_result_tk
    
    label_count.config(text=f"Boundaries found: {num_labels}")





def select_file():
    filename = filedialog.askopenfilename()
    entry_path.delete(0, tk.END)
    entry_path.insert(0, filename)






root = tk.Tk()
root.geometry("600x400")
root.title("Boundary Detection GUI")

frame_controls = tk.Frame(root)
frame_controls.pack()

label_path = tk.Label(frame_controls, text="Image Path:")
label_path.grid(row=0, column=0)
entry_path = tk.Entry(frame_controls, width=50)
entry_path.grid(row=0, column=1)
entry_path.insert(0, "../imgs/Beans_2.bmp")
btn_browse = tk.Button(frame_controls, text="Browse", command=select_file)
btn_browse.grid(row=0, column=2)

label_iterations = tk.Label(frame_controls, text="Erosion Iterations:")
label_iterations.grid(row=1, column=0)
entry_iterations = tk.Entry(frame_controls, width=10)
entry_iterations.grid(row=1, column=1)
entry_iterations.insert(0, "15")

label_thickness = tk.Label(frame_controls, text="Border Thickness:")
label_thickness.grid(row=2, column=0)
entry_thickness = tk.Entry(frame_controls, width=10)
entry_thickness.grid(row=2, column=1)
entry_thickness.insert(0, "1")

btn_process = tk.Button(root, text="Process Image", command=process_image)
btn_process.pack()

frame_images = tk.Frame(root)
frame_images.pack()

label_original = tk.Label(frame_images)
label_original.pack(side=tk.LEFT)

label_result = tk.Label(frame_images)
label_result.pack(side=tk.RIGHT)

label_count = tk.Label(root, text="")
label_count.pack()

root.mainloop()






--- 0.20835590362548828 seconds ---
--- 0.2128896713256836 seconds ---
Circularities per label: {1: np.float64(2.23286220694734), 2: np.float64(0.2555049153020915), 3: np.float64(0.25565782114632135), 4: np.float64(0.14255252914700517), 5: np.float64(1.171793080237294), 6: np.float64(0.06895416614699906), 7: np.float64(0.2583827725039683), 8: np.float64(1.4287579364267533)}
--- 0.14539456367492676 seconds ---
--- 0.14269161224365234 seconds ---
Circularities per label: {1: np.float64(2.0719093966113347), 2: np.float64(0.2348305991944282), 3: np.float64(0.24058347013495623), 4: np.float64(0.2611070924447541), 5: np.float64(1.0983124743069992), 6: np.float64(0.9486789374198686), 7: np.float64(0.24175055041742316), 8: np.float64(1.0392413975897017), 9: np.float64(0.24407102075890677), 10: np.float64(1.2137421444050662)}
